# Ej. 3 · Exploración según la mejora (1,5 puntos)

**Objetivo (enunciado):** que ε no dependa del número de episodios y baje conforme mejora el agente. Hay que responder dos
preguntas: **¿cómo determinamos que el algoritmo ha mejorado?** y **¿cómo trasladamos esa mejora a ε?** El número de
episodios puede seguir fijo.

### Qué cambia respecto al ej. 2

| Bloque | Cambio | Decisión |
|---|---|---|
| Decaimiento de ε | `LinearDecay` (por episodio) → `ImprovementEpsilon` (por evaluación) | D-3.1 |
| Bucle | Evaluación greedy cada 25 episodios en la rejilla fija | D-3.1 |

**¿Cómo medimos la mejora?** Cada 25 episodios se para la exploración y se evalúa la política greedy en los **mismos** 27
episodios de la rejilla (en clase: evaluar sin explorar y en varios episodios, no en uno). La métrica es el error máximo de
seguimiento medio, E, en títulos.

**¿Cómo se traslada a ε?** Progreso = fracción del camino recorrido entre el error sin aprender (E0, primera evaluación) y el
del TWAP determinista (E* ≈ 1): $p = \mathrm{clip}\big((E_0 - E)/(E_0 - E^*), 0, 1\big)$, y
$\varepsilon = \varepsilon_{max} - (\varepsilon_{max} - \varepsilon_{min})\cdot p_{ref}$, con $p_{ref}$ el mejor progreso
alcanzado. Si el agente empeora más de un 10 % en 2 evaluaciones seguidas, $p_{ref}$ baja y **ε vuelve a subir**.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("."))   # ejecutar desde notebooks/
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from common import data as dt, rewards as rw, exploration as ex, protocol as pr, plots as pl
from common.env import BestExecutionEnv
from common.agents import DDQNAgent, TWAPAgent
pd.set_option("display.precision", 3)
%matplotlib inline

In [ ]:
ASSET = "REP"                      # D-0.2
DATA = dt.load(ASSET)
train = DATA["train"]
GRID = pr.make_grid(train, n_days=3, seed=0)   # rejilla fija de evaluación (D-0.3)
print(dt.summary(DATA)); print(len(GRID), "episodios de evaluación")

In [ ]:
CFG = dict(gamma=0.9, alpha=1e-3, buffer_size=50_000, batch_size=64, hidden_neurons=32)
NEP = 300
twap_err = pr.load_df("00_twap_grid").max_err.mean()

def run(schedule, seed):
    pr.set_seeds(seed)
    env = BestExecutionEnv(train, reward=rw.Tracking(), seed=seed)
    agent = DDQNAgent(env, seed=seed, **CFG)
    pr.fill_buffer(agent, env, 25)
    ep, ev = pr.train(agent, env, NEP, schedule, 8, 1000, eval_data=train, eval_grid=GRID, eval_every=25,
                      eval_reward=rw.Tracking(), log_every=0, verbose=False)
    return ep.assign(seed=seed), ev.assign(seed=seed)

RUN = not (pr.RESULTS / "ej3_evals.csv").exists()
if RUN:
    eps_rows, ev_rows = [], []
    for seed in range(3):
        for name, sched in [("lineal", ex.LinearDecay(1.0, 0.05, nepisodes=NEP)),
                            ("mejora", ex.ImprovementEpsilon(target_error=twap_err))]:
            ep, ev = run(sched, seed)
            eps_rows.append(ep.assign(schedule=name)); ev_rows.append(ev.assign(schedule=name))
            print(name, seed, ev.max_err.round(1).tolist())
    pr.save_df(pd.concat(eps_rows), "ej3_train"); pr.save_df(pd.concat(ev_rows), "ej3_evals")
ev3 = pr.load_df("ej3_evals")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for name, g in ev3.groupby("schedule"):
    m = g.groupby("episode")[["max_err", "epsilon"]].mean()
    ax[0].plot(m.index, m.max_err, "o-", label=name); ax[1].plot(m.index, m.epsilon, "o-", label=name)
ax[0].axhline(twap_err, ls="--", c="k", lw=1, label="TWAP determinista")
ax[0].set_title("error máx. de seguimiento (eval. greedy)"); ax[1].set_title("ε en cada evaluación")
for a in ax: a.set_xlabel("episodio"); a.legend(); a.grid(alpha=0.3)
fig.savefig(pr.FIGURES / "ej3_comparacion.png", dpi=130, bbox_inches="tight")

In [ ]:
ok = ev3[ev3.max_err <= 2].groupby(["schedule", "seed"]).episode.min().unstack(0)
print("Primer episodio con error máx. ≤ 2 títulos (por semilla):"); display(ok)
ev3[ev3.episode == NEP].groupby("schedule")[["max_err", "completed"]].agg(["mean", "std"])

## Conclusión

Completar con las cifras: episodios hasta el umbral con cada plan, error final y cuántas veces subió ε. Con el mismo
presupuesto, ε por mejora deja de explorar cuando la política ya sigue el plan y vuelve a explorar si se degrada.